# FLUX.1-dev (fp8) i Colab via TotoroUI
Kör cellerna i ordning. Välj en GPU-runtime (Runtime → Change runtime type).

In [ ]:
%cd /content
!git clone -b totoro3 https://github.com/camenduru/ComfyUI /content/TotoroUI
%cd /content/TotoroUI

!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!pip install -q torchsde einops diffusers accelerate
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install torchsde einops diffusers accelerate
!apt -y install -qq aria2

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/flux1-dev-fp8.safetensors -d /content/TotoroUI/models/unet -o flux1-dev-fp8.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.sft -d /content/TotoroUI/models/vae -o ae.sft
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/clip_l.safetensors -d /content/TotoroUI/models/clip -o clip_l.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/t5xxl_fp8_e4m3fn.safetensors -d /content/TotoroUI/models/clip -o t5xxl_fp8_e4m3fn.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://raw.githubusercontent.com/suegy/tig-330/refs/heads/main/requirements-colab-flux-cuda.txt -d /content -o requirements.txt

import random
import torch
import numpy as np
from PIL import Image

# Ladda requirements om beroenden strular i Colab:
# !pip install -r "/content/requirements_2.txt"

# Skapa en snapshot av installerade paket för att kunna gå tillbaka till en fungerande konfiguration:
# !pip freeze > /content/requirements_2.txt

In [ ]:
import nodes
from nodes import NODE_CLASS_MAPPINGS
from totoro_extras import nodes_custom_sampler
from totoro import model_management

DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
RandomNoise = nodes_custom_sampler.NODE_CLASS_MAPPINGS["RandomNoise"]()
BasicGuider = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicGuider"]()
KSamplerSelect = nodes_custom_sampler.NODE_CLASS_MAPPINGS["KSamplerSelect"]()
BasicScheduler = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicScheduler"]()
SamplerCustomAdvanced = nodes_custom_sampler.NODE_CLASS_MAPPINGS["SamplerCustomAdvanced"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

In [ ]:
import os, inspect
import totoro.sd1_clip as sd1_clip

tok_dir = os.path.join(os.path.dirname(sd1_clip.__file__), "sd1_tokenizer")
print("Using tokenizer dir:", tok_dir)

# Visa om nuvarande merges refererar till comfy</w>
merges_path = os.path.join(tok_dir, "merges.txt")
if os.path.exists(merges_path):
    with open(merges_path, "r", encoding="utf-8", errors="ignore") as f:
        merges_text = f.read()
    print("merges contains 'comfy</w>' ?", "comfy</w>" in merges_text)
else:
    print("merges.txt not found yet")

# Ersätt tokenizer-filerna med en konsekvent CLIP-tokenizer från Hugging Face
from transformers import CLIPTokenizer
tok = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
os.makedirs(tok_dir, exist_ok=True)
tok.save_pretrained(tok_dir)

In [ ]:
with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("t5xxl_fp8_e4m3fn.safetensors", "clip_l.safetensors", "flux")[0]
    unet = UNETLoader.load_unet("flux1-dev-fp8.safetensors", "fp8_e4m3fn")[0]
    vae = VAELoader.load_vae("ae.sft")[0]

def closestNumber(n, m):
    q = int(n / m)
    n1 = m * q
    if (n * m) > 0:
        n2 = m * (q + 1)
    else:
        n2 = m * (q - 1)
    if abs(n - n1) < abs(n - n2):
        return n1
    return n2

## Generera bild

In [ ]:
@torch.inference_mode()
def generate(prompt, width=1024, height=1024, seed=0, steps=20,
             sampler_name="euler", scheduler="simple"):
    if seed == 0:
        seed = random.randint(0, 2**64 - 1)
    print("seed:", seed)

    cond, pooled = clip.encode_from_tokens(clip.tokenize(prompt), return_pooled=True)
    cond = [[cond, {"pooled_output": pooled}]]

    noise = RandomNoise.get_noise(seed)[0]
    guider = BasicGuider.get_guider(unet, cond)[0]
    sampler = KSamplerSelect.get_sampler(sampler_name)[0]
    sigmas = BasicScheduler.get_sigmas(unet, scheduler, steps, 1.0)[0]
    latent = EmptyLatentImage.generate(closestNumber(width, 16), closestNumber(height, 16))[0]

    sample, _ = SamplerCustomAdvanced.sample(noise, guider, sampler, sigmas, latent)
    model_management.soft_empty_cache()

    decoded = VAEDecode.decode(vae, sample)[0].detach()
    img = Image.fromarray(np.array(decoded * 255, dtype=np.uint8)[0])
    img.save("/content/flux.png")
    return img

generate("a cozy cabin in a snowy forest at dusk, warm light in the windows, photorealistic",
         width=1024, height=1024, seed=0, steps=20)

## Gradio-gränssnitt

In [ ]:
!pip install -q gradio

import gradio as gr

def ui_generate(prompt, width, height, seed, steps, sampler_name, scheduler):
    return generate(prompt, int(width), int(height), int(seed), int(steps), sampler_name, scheduler)

with gr.Blocks(title="FLUX.1-dev") as demo:
    gr.Markdown("# FLUX.1-dev (fp8)")
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label="Prompt", lines=4,
                                value="a cozy cabin in a snowy forest at dusk, warm light in the windows, photorealistic")
            with gr.Row():
                width = gr.Slider(256, 2048, value=1024, step=16, label="Bredd")
                height = gr.Slider(256, 2048, value=1024, step=16, label="Höjd")
            steps = gr.Slider(1, 50, value=20, step=1, label="Steg")
            seed = gr.Number(value=0, precision=0, label="Seed (0 = slumpad)")
            sampler_name = gr.Dropdown(["euler", "euler_ancestral", "heun", "dpmpp_2m", "dpmpp_sde", "ddim", "uni_pc"],
                                       value="euler", label="Sampler")
            scheduler = gr.Dropdown(["simple", "normal", "karras", "exponential", "sgm_uniform", "beta"],
                                    value="simple", label="Scheduler")
            btn = gr.Button("Generera", variant="primary")
        with gr.Column():
            out = gr.Image(label="Resultat", type="pil")
    btn.click(ui_generate, [prompt, width, height, seed, steps, sampler_name, scheduler], out)

demo.queue().launch(share=True, debug=True)